# Facial Emotion Recognition - Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

%matplotlib inline
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
DATA_PATH = '../data/raw/fer2013.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../data/processed/fer2013_cleaned.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head(10))
display(df.info())
display(df.describe())

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicated rows:', df.duplicated().sum())
print('\nData types:\n', df.dtypes)

In [ ]:
idx = 0
pixels = np.array(df.loc[idx, 'pixels'].split(), dtype=np.float32)
image = pixels.reshape(48, 48)

plt.imshow(image, cmap='gray')
plt.title(f'Label: {df.loc[idx, "emotion"]}')
plt.axis('off')
plt.show()
print(f'Pixel vector length: {len(pixels)}')
print(f'First 20 pixel values: {pixels[:20]}')

In [ ]:
emotion_map = {
    0: 'Angry',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happy',
    4: 'Sad',
    5: 'Surprise',
    6: 'Neutral'
}

df['emotion_label'] = df['emotion'].map(emotion_map)

plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df, x='emotion', palette='viridis')
ax.set_xticklabels([emotion_map[i] for i in sorted(df['emotion'].unique())])
plt.title('Class Distribution of Emotions in FER2013')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.show()

print('\nClass counts:')
print(df['emotion_label'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (emotion_id, label) in enumerate(emotion_map.items()):
    sample = df[df['emotion'] == emotion_id].iloc[0]
    pixels = np.array(sample['pixels'].split(), dtype=np.float32).reshape(48, 48)
    axes[i].imshow(pixels, cmap='gray')
    axes[i].set_title(label, fontsize=14, fontweight='bold')
    axes[i].axis('off')

axes[7].axis('off')
plt.suptitle('Sample Image per Emotion Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
all_pixels = np.concatenate([
    np.array(row.split(), dtype=np.float32)
    for row in df['pixels'].sample(1000, random_state=42)
])

plt.figure(figsize=(10, 6))
plt.hist(all_pixels, bins=256, color='steelblue', alpha=0.7, edgecolor='black')
plt.title('Pixel Intensity Distribution (Sample of 1000 images)')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
usage_counts = df['Usage'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(usage_counts.values, labels=usage_counts.index, autopct='%1.1f%%',
        startangle=90, colors=['#66b3ff', '#99ff99', '#ffcc99'],
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Dataset Split: Usage Distribution', fontsize=14, fontweight='bold')
plt.show()
print(usage_counts)

In [ ]:
pixel_data = np.array([
    np.array(row.split(), dtype=np.float32)
    for row in df['pixels']
])

mean_val = pixel_data.mean() / 255.0
std_val = pixel_data.std() / 255.0

print(f'Dataset-wide pixel mean (normalized): {mean_val:.4f}')
print(f'Dataset-wide pixel std  (normalized): {std_val:.4f}')
print(f'Dataset-wide pixel mean (raw):       {pixel_data.mean():.2f}')
print(f'Dataset-wide pixel std  (raw):       {pixel_data.std():.2f}')

In [ ]:
sample_size = 500
pixel_sample = pixel_data[:sample_size]
corr_matrix = np.corrcoef(pixel_sample)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0,
            xticklabels=False, yticklabels=False,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title(f'Pixel Correlation Heatmap (Sample of {sample_size} images)', fontsize=14, fontweight='bold')
plt.xlabel('Image Index')
plt.ylabel('Image Index')
plt.tight_layout()
plt.show()

## Summary of Findings

### Class Imbalance
- The dataset exhibits significant class imbalance. 'Happy' is the most frequent class, while 'Disgust' has very few samples.
- This imbalance can bias the model toward majority classes. Data augmentation (horizontal flip, rotation, etc.) or class weighting should be applied during training.

### Image Quality
- Images are 48x48 grayscale with pixel values ranging from 0 to 255.
- Pixel intensity distribution shows a broad spread, indicating diverse lighting and contrast conditions.
- The mean pixel intensity (normalized) is around 0.4–0.5, suggesting reasonably centered data.

### Dataset Split
- The data is pre-split into Training (~70%), PublicTest (~15%), and PrivateTest (~15%) subsets.
- These splits are suitable for training, validation, and final evaluation respectively.

### Correlation
- Low inter-image correlation suggests diverse facial expressions and minimal redundancy in the dataset.

### Next Steps
- Normalize pixel values to [0,1] or standardize using computed mean/std.
- Apply data augmentation to address class imbalance.
- Define train/val/test loaders based on the Usage column.